In [1]:
import warnings
from IPython.display import Markdown, display
warnings.filterwarnings("ignore")

In [2]:
import pdfplumber as pdf

In [3]:
filename = "Muhammad-pages.pdf"
dictionary_for_pages=[]
with pdf.open(f"../data/building/{filename}") as pdfFile:
    for pageNo,content in enumerate(pdfFile.pages):
        dictionary_for_pages.append({"pageNo":pageNo,"text" : content.extract_text()})
        


In [29]:
chunks = []
chunkNumber = 1

heading = "Default Heading"
current_text = []
startPage = None
endPage = None

for singleDictionary in dictionary_for_pages:

    pageNo = singleDictionary["pageNo"]
    pageText = singleDictionary["text"]

    lines = pageText.split("\n")

    for line in lines:
        line = line.strip()

        if not line:
            continue

        cleaned_line = re.sub(r'\[\d+\]', '', line)
        words = cleaned_line.split()

        # heading detected
        if len(words) < 8 and not cleaned_line.endswith("."):

            if current_text:
                chunks.append({
                    "startPage": startPage,
                    "endPage": endPage,
                    "chunkNumber": chunkNumber,
                    "heading": heading,
                    "chunk_text": " ".join(current_text).strip()
                })

                chunkNumber += 1

            heading = line
            current_text = []

            # new chunk starts from current page
            startPage = pageNo

        else:
            # first line of chunk
            if startPage is None:
                startPage = pageNo

            # update every time text is added
            endPage = pageNo

            current_text.append(line)


# push last remaining chunk
if current_text:
    chunks.append({
        "startPage": startPage,
        "endPage": endPage,
        "chunkNumber": chunkNumber,
        "heading": heading,
        "chunk_text": " ".join(current_text).strip()
    })

In [37]:
# chunks

In [22]:
from sentence_transformers import SentenceTransformer as ST
model = ST("all-MiniLM-L6-v2")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 315.36it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
chunksText=[]
for chunk in chunks:
    chunksText.append(chunk["chunk_text"])
chunksText
embeddings = model.encode(chunksText)

In [32]:
embeddings

array([[-0.01479094,  0.15283845,  0.00446697, ..., -0.03008716,
        -0.0549854 ,  0.00385174],
       [ 0.00460467,  0.15764512, -0.04094992, ..., -0.01765833,
        -0.02792246, -0.03608555],
       [-0.01667172,  0.03446221, -0.02355837, ..., -0.06727172,
        -0.04321225, -0.00227335],
       [-0.00614691,  0.05360625, -0.03890175, ..., -0.05852663,
         0.01440349, -0.04789002],
       [-0.03182285,  0.05032117, -0.0300909 , ..., -0.05687704,
         0.03145624, -0.03772489]], shape=(5, 384), dtype=float32)

In [33]:
import faiss 

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [34]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")
os.getenv("GROQ_API_KEY")
# how to create client groq thing
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
print(client)

In [46]:
def ask(query):
    message = (query + "\n Answer the above question,only using the text below ,and \
    say I don't know if not found ,and also if you do find an answer then also tell, \
    which page number and chunk number u used,if startpage and end page are different ,then tell both,otherwise only one ")
    
    for i in indices[0]:
        message +=f"Page Number = {chunks[i]["startPage"]}\n"
        message +=f"Page Number = {chunks[i]["endPage"]}\n"
        message +=f"Chunk Number = {chunks[i]["chunkNumber"]}\n"
        message += chunks[i]["chunk_text"] + "\n"
    messages = [{"role": "user", "content": message}]
    response  = client.chat.completions.create(model="llama-3.3-70b-versatile",messages=messages)
    return response

In [47]:
query1 = "What happened during the Satanic Verses incident?"
query_embeddings =  model.encode([query1])
distances,indices = index.search(query_embeddings,3)
response = ask(query1)
# irrelevant question,to test what it responds
query2 = "What's the capital of France?"
query_embeddings =  model.encode([query2])
distances,indices = index.search(query_embeddings,3)
response1 = ask(query2)

In [48]:
responseOfQuery = response.choices[0].message.content
responseOfQuery1 = response1.choices[0].message.content


In [49]:
display(Markdown(responseOfQuery))

The Satanic Verses incident occurred when Muhammad was said to have been given false verses by Satan, acknowledging pre-Islamic pagan goddesses, which led to a reconciliation between Muhammad and the Meccans. However, the next day, Muhammad retracted and abrogated the verses at the behest of fresh divine revelation from Gabriel. 

I found this information on Page Number = 1, Chunk Number = 3.

In [50]:
display(Markdown(responseOfQuery1))

I don't know. The text does not mention the capital of France.